In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score,classification_report, confusion_matrix
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow")

In [5]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='yt-comment-sentiment-analysis', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/yt-comment-sentiment-analysis"

Repository Aayush10671/yt-comment-sentiment-analysis initialized!

🏃 View run unruly-ape-306 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0/runs/8fb368b4a14e40309a701401b9d96571
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0


In [6]:
df = pd.read_csv("preprocessed_data.csv")
df.shape

(36793, 2)

In [7]:
print(df['clean_comment'].isnull().sum())          # count of NaN
print(df['clean_comment'].dtype)                   # should be object or string
print(df['clean_comment'].apply(type).value_counts())  # see if all are strings
print(df['clean_comment'].str.len().value_counts().head()) # check empty strings

131
object
clean_comment
<class 'str'>      36662
<class 'float'>      131
Name: count, dtype: int64
clean_comment
14.0    439
15.0    429
24.0    421
19.0    420
22.0    416
Name: count, dtype: int64


In [8]:
# Drop rows where clean_comment is missing
df = df.dropna(subset=['clean_comment'])
# Remove rows where the comment is empty after stripping
df = df[df['clean_comment'].str.strip() != '']
# Ensure all values are strings (just in case)
df['clean_comment'] = df['clean_comment'].astype(str)

In [9]:
df = df.dropna(subset=['clean_comment'])
df = df[df['clean_comment'].str.strip() != '']
df['clean_comment'] = df['clean_comment'].astype(str)

In [10]:
mlflow.set_experiment("exp-5 best model")

2026/07/25 23:33:33 INFO mlflow.tracking.fluent: Experiment with name 'exp-5 best model' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/267af5ba92e34d958dae26bbcd71ef72', creation_time=1785002614298, experiment_id='6', last_update_time=1785002614298, lifecycle_stage='active', name='exp-5 best model', tags={}, workspace='default'>

In [11]:
!pip install optuna

Defaulting to user installation because normal site-packages is not writeable


In [12]:
import optuna
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.naive_bayes import MultinomialNB

In [13]:
df['category'] = df['category'].map({0:0 , 1:1 , -1:2})


In [14]:
df.head(5)

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,2
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [16]:
ngram_range = (1,3)
max_feature = 1000
vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_feature
    )

X = vectorizer.fit_transform(df["clean_comment"])
y = df["category"].values

smote = SMOTE(random_state = 42)

X_resampled, y_resampled = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
        X_resampled,
        y_resampled,
        test_size=0.2,
        random_state=42,
    )

def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):

    with mlflow.start_run():

        
        mlflow.set_tag("Model", model_name)
        mlflow.set_tag("Vectorizer", "TF-IDF")
        mlflow.set_tag("Sampling", "SMOTE")

        mlflow.log_param("ngram_range", (1, 3))
        mlflow.log_param("max_features", 1000)

        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)

        if hasattr(model, "max_depth"):
            mlflow.log_param("max_depth", model.max_depth)

        if hasattr(model, "learning_rate"):
            mlflow.log_param("learning_rate", model.learning_rate)

        if hasattr(model, "n_neighbors"):
            mlflow.log_param("n_neighbors", model.n_neighbors)

      
        model.fit(X_train, y_train)

       
        y_pred = model.predict(X_test)

        
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in report.items():

            if isinstance(metrics, dict):

                for metric_name, metric_value in metrics.items():

                    mlflow.log_metric(
                        f"{label}_{metric_name}",
                        metric_value
                    )

   
        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"{model_name} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        plt.savefig("confusion_matrix.png")
        plt.close()

        mlflow.log_artifact("confusion_matrix.png")

     
        with open("classification_report.txt", "w") as f:
            f.write(classification_report(y_test, y_pred))

        mlflow.log_artifact("classification_report.txt")

        mlflow.sklearn.log_model(
            sk_model=model,
            name=model_name
        )

        print("=" * 50)
        print(f"Model    : {model_name}")
        print(f"Accuracy : {accuracy:.4f}")
        print("=" * 50)


In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=15,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss"
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        max_depth=15,
        learning_rate=0.1,
        random_state=42,
        verbosity=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "NaiveBayes": MultinomialNB()
}

In [18]:
for model_name, model in models.items():
    log_mlflow(
        model_name,
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

2026/07/25 23:56:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model    : RandomForest
Accuracy : 0.6990
🏃 View run upbeat-fish-381 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/cc58ce6ca1d44de4a5c2f140501c2a37
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6


2026/07/25 23:59:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model    : XGBoost
Accuracy : 0.8094
🏃 View run overjoyed-snipe-773 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/9a169dde63094538a816152a5680b824
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6


C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/26 00:01:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model    : LightGBM
Accuracy : 0.8122
🏃 View run resilient-bass-499 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/27ca7a2a2d68440286caebae9f85db98
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6
🏃 View run angry-goat-888 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/b932aa5f7ecf4ff49740e5061655732c
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/6


KeyboardInterrupt: 